# Triage workflow evaluation candidates

This notebook checks local model parity and runs a small synthetic retrieval smoke evaluation through the production workflow. It creates a separate LangSmith smoke dataset and experiment in later cells.

The score references come from `tests/fixtures/val_reference_sample.json`; they are local-only and are not uploaded. Retrieval examples use synthetic requests and canonical shape labels. This is a smoke test, not an independent benchmark. No generated-answer evaluation or RAGAS applies to this deterministic workflow. The final test partition is not used.

## Evaluation scope

Inputs are allowlisted intake categories, calendar features, optional borough filter, and synthetic time. Expected values are workflow status, cutoff flag, local parity score, and canonical shape labels.

Do not include ticket IDs, addresses, coordinates, ZIP codes, free text, raw Qdrant payloads, embeddings, credentials, or final-test data. Validation-derived parity examples stay local; hosted smoke examples are synthetic.

In [1]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name != "resolveiq":
    REPO_ROOT = next(parent for parent in REPO_ROOT.parents if (parent / "pyproject.toml").exists())

PARITY_FIXTURE = REPO_ROOT / "tests" / "fixtures" / "val_reference_sample.json"
THIS_NOTEBOOK = "notebooks/04_evaluation_design/build_triage_langsmith_reference_set.ipynb"

print(f"repo             {REPO_ROOT}")
print(f"parity fixture   {PARITY_FIXTURE}")
print(f"candidate output {THIS_NOTEBOOK} only; no file will be written")

repo             C:\Users\mejdi\Documents\resolveiq
parity fixture   C:\Users\mejdi\Documents\resolveiq\tests\fixtures\val_reference_sample.json
candidate output notebooks/04_evaluation_design/build_triage_langsmith_reference_set.ipynb only; no file will be written


## Candidate schema

Each record stores provenance, a structured workflow input, expected behavior, and review status:

| Group | Purpose |
| --- | --- |
| `example_key` and `source_kind` | Stable local handle and provenance category |
| `input` | Safe fields accepted by the triage workflow |
| `expected` | Behavior that a later local test or experiment may check |
| `review` | Human decision, provenance, and limitations before export |

`validation_parity_reference` records remain local-only. `proposed_synthetic_retrieval_seed` records are smoke-test labels, not row- or pilot-derived ground truth. Hosted projections contain canonical shapes only—never ticket IDs or raw payloads.

In [2]:
SAFE_INPUT_FIELDS = {
    "complaint_type",
    "location_type",
    "descriptor",
    "descriptor_2",
    "open_data_channel_type",
    "agency",
    "created_hour",
    "created_day_of_week",
    "created_is_weekend",
    "borough",
    "intake_time",
    "limit",
    "include_similar_cases",
}
SAFE_EXPECTED_FIELDS = {
    "risk_score_reference",
    "risk_score_tolerance",
    "expected_operating_cutoff",
    "expected_above_frozen_cutoff",
    "expected_routing_decision",
    "expected_retrieval_status",
    "expected_shape_slugs",
    "expected_shape_hit_k",
    "proposed_expected_shape_slugs",
    "proposed_shape_hit_k",
    "expected_borough_filter_match",
    "expected_temporal_rule",
}
FORBIDDEN_FIELD_NAMES = {
    "ticket",
    "unique_key",
    "address",
    "incident_zip",
    "latitude",
    "longitude",
    "resolution_description",
    "payload",
    "embedding",
}

print(f"safe input fields: {len(SAFE_INPUT_FIELDS)}")
print(f"safe expected fields: {len(SAFE_EXPECTED_FIELDS)}")
print(f"forbidden field names: {sorted(FORBIDDEN_FIELD_NAMES)}")

safe input fields: 13
safe expected fields: 12
forbidden field names: ['address', 'embedding', 'incident_zip', 'latitude', 'longitude', 'payload', 'resolution_description', 'ticket', 'unique_key']


## Local model-parity references

These three score-only records match the first three entries in `tests/fixtures/val_reference_sample.json`. They verify frozen-artifact parity; they are not synthetic examples or predictive-quality measurements and must not be uploaded. Retrieval is disabled, so the expected result is `similar_cases=null` and `retrieval_status=not_requested`.

In [3]:
CANDIDATE_EXAMPLES = [
    {
        "example_key": "validation-parity-001",
        "source_kind": "validation_parity_reference",
        "input": {
            "descriptor": "APARTMENT ONLY",
            "descriptor_2": "NO HEAT",
            "location_type": "RESIDENTIAL BUILDING",
            "open_data_channel_type": "ONLINE",
            "agency": "HPD",
            "created_hour": 18,
            "created_day_of_week": 0,
            "created_is_weekend": 0,
            "include_similar_cases": False,
        },
        "expected": {
            "risk_score_reference": 0.13472128017204477,
            "risk_score_tolerance": 1e-6,
            "expected_operating_cutoff": 0.28,
            "expected_above_frozen_cutoff": False,
            "expected_routing_decision": "scored_only",
            "expected_retrieval_status": "not_requested",
        },
        "review": {
            "status": "local_only_do_not_export",
            "provenance_file": "tests/fixtures/val_reference_sample.json",
            "provenance_record_index": 0,
            "note": "Validation parity reference only; not synthetic or an accuracy measurement.",
        },
    },
    {
        "example_key": "validation-parity-002",
        "source_kind": "validation_parity_reference",
        "input": {
            "descriptor": "APARTMENT ONLY",
            "descriptor_2": "NO HEAT",
            "location_type": "RESIDENTIAL BUILDING",
            "open_data_channel_type": "PHONE",
            "agency": "HPD",
            "created_hour": 21,
            "created_day_of_week": 0,
            "created_is_weekend": 0,
            "include_similar_cases": False,
        },
        "expected": {
            "risk_score_reference": 0.14855617188839787,
            "risk_score_tolerance": 1e-6,
            "expected_operating_cutoff": 0.28,
            "expected_above_frozen_cutoff": False,
            "expected_routing_decision": "scored_only",
            "expected_retrieval_status": "not_requested",
        },
        "review": {
            "status": "local_only_do_not_export",
            "provenance_file": "tests/fixtures/val_reference_sample.json",
            "provenance_record_index": 1,
            "note": "Validation parity reference only; not synthetic or an accuracy measurement.",
        },
    },
    {
        "example_key": "validation-parity-003",
        "source_kind": "validation_parity_reference",
        "input": {
            "descriptor": "APARTMENT ONLY",
            "descriptor_2": "NO HEAT",
            "location_type": "RESIDENTIAL BUILDING",
            "open_data_channel_type": "ONLINE",
            "agency": "HPD",
            "created_hour": 14,
            "created_day_of_week": 0,
            "created_is_weekend": 0,
            "include_similar_cases": False,
        },
        "expected": {
            "risk_score_reference": 0.1358251505430689,
            "risk_score_tolerance": 1e-6,
            "expected_operating_cutoff": 0.28,
            "expected_above_frozen_cutoff": False,
            "expected_routing_decision": "scored_only",
            "expected_retrieval_status": "not_requested",
        },
        "review": {
            "status": "local_only_do_not_export",
            "provenance_file": "tests/fixtures/val_reference_sample.json",
            "provenance_record_index": 2,
            "note": "Validation parity reference only; not synthetic or an accuracy measurement.",
        },
    },
]

VALIDATION_PARITY_RECORDS = json.loads(PARITY_FIXTURE.read_text(encoding="utf-8"))
assert len(VALIDATION_PARITY_RECORDS) == 5
for example in CANDIDATE_EXAMPLES:
    record_index = example["review"]["provenance_record_index"]
    source_record = VALIDATION_PARITY_RECORDS[record_index]
    model_input_fields = {
        key: value for key, value in example["input"].items() if key != "include_similar_cases"
    }
    assert source_record["features"] == model_input_fields
    assert example["input"]["include_similar_cases"] is False
    assert abs(source_record["expected_score"] - example["expected"]["risk_score_reference"]) < 1e-12

print(f"candidate examples: {len(CANDIDATE_EXAMPLES)}")
print("PASS: the three score examples are explicitly verified against the local validation fixture.")
print("PASS: model fields match exactly and include_similar_cases is separately verified as False.")
print("PASS: these validation-derived examples are marked local_only_do_not_export.")

candidate examples: 3
PASS: the three score examples are explicitly verified against the local validation fixture.
PASS: model fields match exactly and include_similar_cases is separately verified as False.
PASS: these validation-derived examples are marked local_only_do_not_export.


## Synthetic retrieval smoke seeds

These two structured requests use proposed canonical-shape labels. They are not copies of pilot rows or queries. The eight-query pilot compared two complete retrieval designs (Hit@1: 4/8 vs 7/8; Hit@3: 8/8 for both); it does not validate these labels.

Smoke evaluation uses canonical shapes only. No ticket-ID mapping or historical payload is needed or uploaded.

In [4]:
CANDIDATE_EXAMPLES.extend([
    {
        "example_key": "proposed-synthetic-retrieval-001",
        "source_kind": "proposed_synthetic_retrieval_seed",
        "input": {
            "complaint_type": "HEAT/HOT WATER",
            "location_type": "RESIDENTIAL BUILDING",
            "descriptor": "APARTMENT ONLY",
            "descriptor_2": "NO HEAT",
            "borough": "BROOKLYN",
            "intake_time": "2024-01-31T12:00:00",
            "limit": 5,
            "include_similar_cases": True,
        },
        "expected": {
            "expected_retrieval_status": "succeeded",
            "proposed_expected_shape_slugs": ["APT_NO_HEAT"],
            "proposed_shape_hit_k": 1,
            "expected_borough_filter_match": True,
            "expected_temporal_rule": "created_date < intake_time; expose outcomes only when closed_date < intake_time",
        },
        "review": {
            "status": "approved_for_smoke_only",
            "note": "Approved only for the existing two-example smoke check; not an independent relevance benchmark.",
        },
    },
    {
        "example_key": "proposed-synthetic-retrieval-002",
        "source_kind": "proposed_synthetic_retrieval_seed",
        "input": {
            "complaint_type": "HEAT/HOT WATER",
            "location_type": "RESIDENTIAL BUILDING",
            "descriptor": "ENTIRE BUILDING",
            "descriptor_2": "NO HOT WATER",
            "borough": "QUEENS",
            "intake_time": "2024-01-31T12:00:00",
            "limit": 5,
            "include_similar_cases": True,
        },
        "expected": {
            "expected_retrieval_status": "succeeded",
            "proposed_expected_shape_slugs": ["BLDG_NO_HOT"],
            "proposed_shape_hit_k": 1,
            "expected_borough_filter_match": True,
            "expected_temporal_rule": "created_date < intake_time; expose outcomes only when closed_date < intake_time",
        },
        "review": {
            "status": "approved_for_smoke_only",
            "note": "The canonical shape catalog maps ENTIRE BUILDING + NO HOT WATER to BLDG_NO_HOT. One exploratory retrieval returned four BLDG_NO_HOT and one BLDG_NO_HEAT in the top five. Approved only for a two-example smoke check; this does not establish relevance.",
        },
    },
])

print(f"candidate examples after proposed retrieval seeds: {len(CANDIDATE_EXAMPLES)}")

candidate examples after proposed retrieval seeds: 5


## Local safety checks

The structural gate checks field allowlists, forbidden data, and provenance before hosted evaluation. It makes no network call. Review the synthetic labels before enabling the later LangSmith dataset and experiment cells.

In [ ]:
def flattened_keys(value):
    if isinstance(value, dict):
        keys = set(value)
        for nested in value.values():
            keys.update(flattened_keys(nested))
        return keys
    if isinstance(value, list):
        keys = set()
        for item in value:
            keys.update(flattened_keys(item))
        return keys
    return set()

assert CANDIDATE_EXAMPLES
for example in CANDIDATE_EXAMPLES:
    assert example["example_key"]
    assert example["source_kind"] in {
        "validation_parity_reference",
        "proposed_synthetic_retrieval_seed",
    }
    assert example["review"]["status"] in {
        "local_only_do_not_export",
        "approved_for_smoke_only",
    }
    assert set(example["input"]).issubset(SAFE_INPUT_FIELDS)
    assert set(example["expected"]).issubset(SAFE_EXPECTED_FIELDS)
    assert flattened_keys(example).isdisjoint(FORBIDDEN_FIELD_NAMES)

print("PASS: candidate inputs and expected fields use only the declared safe schema.")
print("PASS: validation-derived examples remain local-only; each retrieval set is limited to its stated smoke scope.")
print("PASS: this cell performs local schema checks only; hosted actions are in separate explicit opt-in cells.")

## Review before a hosted run

Check the shape labels against the six-shape catalog, verify `created_date < intake_time` and `closed_date < intake_time`, and confirm there are no ticket IDs, payloads, free text, precise geography, or embeddings. The validation parity examples must remain local.

The later cells create/reuse a separate synthetic smoke dataset and run its experiment; do not run them until labels are approved. No generated answers or final-test data belong in this evaluation.

## Candidate Set v2: six canonical shapes, two borough contexts each

These 12 synthetic rows cover every shape in Notebook 02's six-shape catalog twice. The second row for each shape changes the borough filter and intake time, while keeping the canonical descriptors fixed. `confusable_neighbors` identifies nearby catalog shapes for human review; it is not a negative label or a claim about retrieved results.

Local smoke check completed: all 12 expected canonical shapes appeared at Hit@1 and in the top five. This supports these labels for smoke testing only; it is not an independent relevance benchmark because inputs exactly match the six canonical descriptor combinations.

The production retrieval request is structured. Its embedding text uses complaint type, location type, and the two descriptors; borough and time are not embedded. So this is six distinct embedding queries across two filter contexts each, not 12 paraphrase tests. The recorded local check used intake_time=2024-02-05T12:00:00 for all 12 requests, so it did not separately test both candidate timestamps. The upcoming LangSmith experiment will use each example's own timestamp.

The examples are approved for smoke evaluation only. Their contrast shapes are not negative labels or claims about general ranking quality. No ticket IDs, payloads, or resolution text are included.

In [ ]:
from collections import Counter

CANONICAL_SHAPE_BY_DESCRIPTORS = {
    ("APARTMENT ONLY", "NO HEAT"): "APT_NO_HEAT",
    ("APARTMENT ONLY", "NO HEAT AND NO HOT WATER"): "APT_BOTH",
    ("APARTMENT ONLY", "NO HOT WATER"): "APT_NO_HOT",
    ("ENTIRE BUILDING", "NO HEAT"): "BLDG_NO_HEAT",
    ("ENTIRE BUILDING", "NO HEAT AND NO HOT WATER"): "BLDG_BOTH",
    ("ENTIRE BUILDING", "NO HOT WATER"): "BLDG_NO_HOT",
}
CONFUSABLE_NEIGHBORS = {
    "APT_NO_HEAT": ["APT_BOTH", "APT_NO_HOT"],
    "APT_BOTH": ["APT_NO_HEAT", "APT_NO_HOT"],
    "APT_NO_HOT": ["APT_BOTH", "APT_NO_HEAT"],
    "BLDG_NO_HEAT": ["BLDG_BOTH", "BLDG_NO_HOT"],
    "BLDG_BOTH": ["BLDG_NO_HEAT", "BLDG_NO_HOT"],
    "BLDG_NO_HOT": ["BLDG_BOTH", "BLDG_NO_HEAT"],
}

V2_CASE_ROWS = [
    ("retrieval-v2-01", "APARTMENT ONLY", "NO HEAT", "BROOKLYN", "2024-02-01T12:00:00"),
    ("retrieval-v2-02", "APARTMENT ONLY", "NO HEAT", "QUEENS", "2024-02-05T12:00:00"),
    ("retrieval-v2-03", "APARTMENT ONLY", "NO HEAT AND NO HOT WATER", "BRONX", "2024-02-01T12:00:00"),
    ("retrieval-v2-04", "APARTMENT ONLY", "NO HEAT AND NO HOT WATER", "MANHATTAN", "2024-02-05T12:00:00"),
    ("retrieval-v2-05", "APARTMENT ONLY", "NO HOT WATER", "QUEENS", "2024-02-01T12:00:00"),
    ("retrieval-v2-06", "APARTMENT ONLY", "NO HOT WATER", "STATEN ISLAND", "2024-02-05T12:00:00"),
    ("retrieval-v2-07", "ENTIRE BUILDING", "NO HEAT", "MANHATTAN", "2024-02-01T12:00:00"),
    ("retrieval-v2-08", "ENTIRE BUILDING", "NO HEAT", "BRONX", "2024-02-05T12:00:00"),
    ("retrieval-v2-09", "ENTIRE BUILDING", "NO HEAT AND NO HOT WATER", "BROOKLYN", "2024-02-01T12:00:00"),
    ("retrieval-v2-10", "ENTIRE BUILDING", "NO HEAT AND NO HOT WATER", "STATEN ISLAND", "2024-02-05T12:00:00"),
    ("retrieval-v2-11", "ENTIRE BUILDING", "NO HOT WATER", "QUEENS", "2024-02-01T12:00:00"),
    ("retrieval-v2-12", "ENTIRE BUILDING", "NO HOT WATER", "MANHATTAN", "2024-02-05T12:00:00"),
]

RETRIEVAL_CANDIDATES_V2 = []
for example_key, descriptor, descriptor_2, borough, intake_time in V2_CASE_ROWS:
    expected_slug = CANONICAL_SHAPE_BY_DESCRIPTORS[(descriptor, descriptor_2)]
    RETRIEVAL_CANDIDATES_V2.append({
        "example_key": example_key,
        "source_kind": "synthetic_canonical_shape_candidate",
        "input": {
            "complaint_type": "HEAT/HOT WATER",
            "location_type": "RESIDENTIAL BUILDING",
            "descriptor": descriptor,
            "descriptor_2": descriptor_2,
            "borough": borough,
            "intake_time": intake_time,
            "limit": 5,
            "include_similar_cases": True,
        },
        "expected": {"expected_shape_slugs": [expected_slug]},
        "review": {
            "status": "approved_for_smoke_only",
            "label_basis": "Notebook 02 canonical descriptor-to-shape catalog",
            "confusable_neighbors": CONFUSABLE_NEIGHBORS[expected_slug],
            "note": "Expected shape was observed at Hit@1 and in top five in local canonical-input retrieval smoke checks; not an independent benchmark.",
        },
    })

# Sanitized observations from the approved local retrieval smoke check.
LOCAL_RETRIEVAL_TOP5_SHAPES_V2 = {
    "retrieval-v2-01": ["APT_NO_HEAT"] * 5,
    "retrieval-v2-02": ["APT_NO_HEAT"] * 5,
    "retrieval-v2-03": ["APT_BOTH"] * 5,
    "retrieval-v2-04": ["APT_BOTH"] * 5,
    "retrieval-v2-05": ["APT_NO_HOT"] * 3 + ["APT_NO_HEAT"] * 2,
    "retrieval-v2-06": ["APT_NO_HOT", "APT_NO_HEAT", "APT_NO_HEAT", "APT_NO_HEAT", "APT_BOTH"],
    "retrieval-v2-07": ["BLDG_NO_HEAT"] * 5,
    "retrieval-v2-08": ["BLDG_NO_HEAT"] * 5,
    "retrieval-v2-09": ["BLDG_BOTH"] * 5,
    "retrieval-v2-10": ["BLDG_BOTH"] * 5,
    "retrieval-v2-11": ["BLDG_NO_HOT"] * 4 + ["BLDG_NO_HEAT"],
    "retrieval-v2-12": ["BLDG_NO_HOT"] * 5,
}
LOCAL_RETRIEVAL_ALL_HISTORICAL_V2 = True  # all 60 returned cases passed the local temporal evidence check
expected_slugs = set(CANONICAL_SHAPE_BY_DESCRIPTORS.values())
assert len(RETRIEVAL_CANDIDATES_V2) == 12
assert len({row["example_key"] for row in RETRIEVAL_CANDIDATES_V2}) == 12
assert set(Counter(row["expected"]["expected_shape_slugs"][0] for row in RETRIEVAL_CANDIDATES_V2).values()) == {2}
assert {row["expected"]["expected_shape_slugs"][0] for row in RETRIEVAL_CANDIDATES_V2} == expected_slugs
assert all(set(row["input"]).issubset(SAFE_INPUT_FIELDS) for row in RETRIEVAL_CANDIDATES_V2)
assert all(set(row["expected"]).issubset(SAFE_EXPECTED_FIELDS) for row in RETRIEVAL_CANDIDATES_V2)
assert all(flattened_keys(row).isdisjoint(FORBIDDEN_FIELD_NAMES) for row in RETRIEVAL_CANDIDATES_V2)
assert set(LOCAL_RETRIEVAL_TOP5_SHAPES_V2) == {row["example_key"] for row in RETRIEVAL_CANDIDATES_V2}
assert all(LOCAL_RETRIEVAL_TOP5_SHAPES_V2[row["example_key"]][0] == row["expected"]["expected_shape_slugs"][0] for row in RETRIEVAL_CANDIDATES_V2)
assert all(row["expected"]["expected_shape_slugs"][0] in LOCAL_RETRIEVAL_TOP5_SHAPES_V2[row["example_key"]][:5] for row in RETRIEVAL_CANDIDATES_V2)
assert LOCAL_RETRIEVAL_ALL_HISTORICAL_V2 is True
assert all(row["review"]["status"] == "approved_for_smoke_only" for row in RETRIEVAL_CANDIDATES_V2)
assert not any(row["source_kind"] == "synthetic_canonical_shape_candidate" for row in CANDIDATE_EXAMPLES)  # keep v2 out of the existing smoke upload path
print("candidate count:", len(RETRIEVAL_CANDIDATES_V2))
print("per-shape count:", dict(Counter(row["expected"]["expected_shape_slugs"][0] for row in RETRIEVAL_CANDIDATES_V2)))
print("review status: all 12 approved for smoke evaluation only")
print("local retrieval result: expected shape Hit@1=12/12 and Hit@5=12/12; all 60 results passed historical eligibility")
print("embedding diversity: six canonical descriptor combinations; borough/time vary only for filter-context review")
for row in RETRIEVAL_CANDIDATES_V2:
    features = row["input"]
    expected = row["expected"]["expected_shape_slugs"][0]
    neighbors = ", ".join(row["review"]["confusable_neighbors"])
    print(f"{row['example_key']} | {features['descriptor']} + {features['descriptor_2']} | {features['borough']} | {expected} | contrast: {neighbors}")

## Local LangSmith payload preview — v2 only, no upload

This preview contains the 12 synthetic v2 cases only. The existing v1 two-example dataset remains unchanged. Validation-derived score references and the earlier two smoke seeds are excluded. Inputs contain only allowlisted categories, borough filters, and synthetic intake times; references are canonical shape labels.

This is a local preview, not an upload action. The v2 cases are approved for smoke evaluation only, based on the exact canonical-input local retrieval check; this is not an independent relevance benchmark.

In [ ]:
hosted_dataset_preview_v2 = []
for example in RETRIEVAL_CANDIDATES_V2:
    assert example["review"]["status"] == "approved_for_smoke_only"
    hosted_dataset_preview_v2.append({
        "inputs": example["input"],
        "outputs": {
            "relevant_shape_slugs": example["expected"]["expected_shape_slugs"],
            "reference_hit_k": 1,
        },
        "metadata": {
            "example_key": example["example_key"],
            "review_status": "approved_for_smoke_only",
            "evaluation_scope": "synthetic_canonical_shape_smoke_only",
        },
    })

assert len(hosted_dataset_preview_v2) == 12
assert all(set(row["inputs"]).issubset(SAFE_INPUT_FIELDS) for row in hosted_dataset_preview_v2)
assert all("ticket" not in flattened_keys(row) and "unique_key" not in flattened_keys(row) for row in hosted_dataset_preview_v2)
print(json.dumps(hosted_dataset_preview_v2, indent=2))
print("PASS: v2 preview only; no LangSmith client, dataset creation, or upload.")

## Create the separate v2 LangSmith smoke dataset

This cell creates or exactly verifies `resolveiq-triage-retrieval-smoke-v2`; it never modifies the existing v1 two-example dataset. Creation is disabled by default; the one-time approved run explicitly enables it. This remains a synthetic canonical-input smoke set, not an independent or representative retrieval benchmark. The three validation-derived score references are excluded.

The upload contains only the structured synthetic request fields and canonical-shape reference labels. It contains no ticket IDs, historical case payloads, free text, or outcomes. The cell reuses an existing dataset only if its examples match exactly; it stops rather than overwrite a mismatch.

In [ ]:
DATASET_NAME = "resolveiq-triage-retrieval-smoke-v2"
RUN_HOSTED_DATASET_CHECK = False  # explicit opt-in; default Run All makes no LangSmith request
ALLOW_HOSTED_DATASET_CREATE = False  # separate explicit opt-in for a new hosted dataset upload
client = None
dataset = None

if RUN_HOSTED_DATASET_CHECK:
    from langsmith import Client
    from resolveiq.config import get_settings

    settings = get_settings()
    if settings.langsmith_api_key is None:
        raise RuntimeError("LangSmith API key is not configured in the local environment.")
    client = Client(
        api_url=settings.langsmith_endpoint,
        api_key=settings.langsmith_api_key.get_secret_value(),
    )
    langsmith_examples = hosted_dataset_preview_v2
    assert len(langsmith_examples) == 12
    assert all("relevant_shape_slugs" in row["outputs"] for row in langsmith_examples)

    if client.has_dataset(dataset_name=DATASET_NAME):
        dataset = client.read_dataset(dataset_name=DATASET_NAME)
        existing = list(client.list_examples(dataset_id=dataset.id, limit=100))
        canonical = lambda rows: sorted(
            json.dumps({"inputs": row["inputs"], "outputs": row["outputs"]}, sort_keys=True)
            for row in rows
        )
        existing_rows = [{"inputs": item.inputs, "outputs": item.outputs} for item in existing]
        if canonical(existing_rows) != canonical(langsmith_examples):
            raise RuntimeError("A LangSmith dataset with this name exists but does not match; no changes were made.")
        dataset_action = "reused exact existing dataset"
    elif ALLOW_HOSTED_DATASET_CREATE:
        dataset = client.create_dataset(
            dataset_name=DATASET_NAME,
            description=(
                "Twelve synthetic ResolveIQ canonical-shape retrieval smoke examples."
                "Not an independent or representative relevance benchmark."
            ),
        )
        client.create_examples(dataset_id=dataset.id, examples=langsmith_examples)
        dataset_action = "created and uploaded exactly twelve synthetic examples"
    else:
        dataset_action = "creation skipped; set ALLOW_HOSTED_DATASET_CREATE = True only for an approved upload"
    print(f"{dataset_action}: {DATASET_NAME} (12 examples)")
else:
    print("Skipped LangSmith dataset check; no credentials loaded or requests made.")

## Run the sanitized v2 smoke experiment against the production workflow

The target calls ResolveIQ's existing `run_triage_workflow`; it does not reimplement scoring or retrieval. It derives the model's NYC calendar features from the supplied synthetic intake timestamp, matching the UI boundary.

Before LangSmith receives each run result, the target reduces retrieved cases to canonical shape slugs and boolean contract checks. Ticket IDs, timestamps from retrieved rows, borough values from retrieved rows, status, closure dates, and resolution text are never returned by the evaluation target. The app's existing trace wrapper also hides graph inputs and outputs. The v2 experiment evaluates 12 cases with three deterministic evaluators at concurrency 1; each target run requests an embedding from Gemini and reads local Qdrant. It writes no Qdrant data.

### Completed smoke run — 2026-09-27

Created the separate 12-example dataset `resolveiq-triage-retrieval-smoke-v2` and completed experiment `resolveiq-triage-smoke-v2-46236127`. The existing v1 two-example dataset was not changed. See the [LangSmith experiment results](https://smith.langchain.com/o/eebf09d2-39b3-4c46-a64b-8e37d43cc31f/datasets/d08ddce5-02ab-4052-a740-477760c4ec15/compare?selectedSessions=91c69aba-1e39-4df3-9669-21d0ed114eae).

Interpret this only as a synthetic canonical-shape smoke test, not an independent retrieval benchmark.

In [ ]:
from zoneinfo import ZoneInfo
from langsmith import evaluate
from resolveiq.cases import parse_and_normalize_timestamp
from resolveiq.workflow import TriageWorkflowRequest, run_triage_workflow

NYC_TZ = ZoneInfo("America/New_York")

def safe_langsmith_target(inputs: dict) -> dict:
    intake_utc, intake_ambiguous = parse_and_normalize_timestamp(inputs["intake_time"])
    if intake_ambiguous:
        raise ValueError("Ambiguous synthetic intake timestamp is not allowed.")
    local_intake = intake_utc.astimezone(NYC_TZ)
    request_data = dict(inputs)
    request_data.update({
        "created_hour": local_intake.hour,
        "created_day_of_week": local_intake.weekday(),
        "created_is_weekend": int(local_intake.weekday() in (5, 6)),
    })
    request = TriageWorkflowRequest.model_validate(request_data)
    result = run_triage_workflow(request.model_dump())
    cases = result.get("similar_cases") or []

    created_before_intake = True
    outcome_masking_pass = True
    for case in cases:
        created_raw = case.get("created_date")
        if not created_raw:
            created_before_intake = False
        else:
            created_utc, created_ambiguous = parse_and_normalize_timestamp(created_raw)
            created_before_intake &= not created_ambiguous and created_utc < intake_utc
        historical = case.get("is_historical_evidence") is True
        closed_raw = case.get("closed_date")
        if historical:
            if not closed_raw:
                outcome_masking_pass = False
            else:
                closed_utc, closed_ambiguous = parse_and_normalize_timestamp(closed_raw)
                outcome_masking_pass &= not closed_ambiguous and closed_utc < intake_utc
        else:
            outcome_masking_pass &= all(
                case.get(field) is None
                for field in ("status", "closed_date", "resolution_description")
            )

    # Return only synthetic-safe workflow summaries; never return retrieved row payloads.
    return {
        "routing_decision": result.get("routing_decision"),
        "retrieval_status": result.get("retrieval_status"),
        "shape_slugs": [case.get("shape_slug") for case in cases],
        "borough_filter_pass": all(case.get("borough") == inputs["borough"] for case in cases),
        "created_before_intake_pass": created_before_intake,
        "outcome_masking_pass": outcome_masking_pass,
    }

def shape_hit_at_1(run, example):
    actual = (run.outputs or {}).get("shape_slugs", [])
    relevant = (example.outputs or {}).get("relevant_shape_slugs", [])
    return {"key": "shape_hit_at_1", "score": float(bool(set(actual[:1]) & set(relevant)))}

def shape_hit_at_5(run, example):
    actual = (run.outputs or {}).get("shape_slugs", [])
    relevant = (example.outputs or {}).get("relevant_shape_slugs", [])
    return {"key": "shape_hit_at_5", "score": float(bool(set(actual[:5]) & set(relevant)))}

def workflow_safety_contract(run, example):
    output = run.outputs or {}
    passed = (
        output.get("routing_decision") == "scored_and_retrieved_precedents"
        and output.get("retrieval_status") == "succeeded"
        and output.get("borough_filter_pass") is True
        and output.get("created_before_intake_pass") is True
        and output.get("outcome_masking_pass") is True
    )
    return {"key": "workflow_safety_contract", "score": float(passed)}

RUN_HOSTED_EXPERIMENT = False  # explicit opt-in; Run All will not launch another experiment
if RUN_HOSTED_EXPERIMENT:
    if dataset is None:
        raise RuntimeError("No verified dataset is available; create/verify it first.")
    experiment_results = client.evaluate(
        safe_langsmith_target,
        data=DATASET_NAME,
        evaluators=[shape_hit_at_1, shape_hit_at_5, workflow_safety_contract],
        experiment_prefix="resolveiq-triage-smoke-v2",
        description=(
            "Twelve synthetic canonical retrieval cases; smoke evaluation only."
            "References match canonical descriptor shapes and are not an independent benchmark."
        ),
        metadata={"evaluation_scope": "twelve_example_canonical_synthetic_smoke_only", "test_partition_accessed": False},
        max_concurrency=1,
        num_repetitions=1,
        blocking=True,
    )
    print("Experiment complete: twelve sanitized runs, three deterministic evaluators each.")
else:
    experiment_results = None
    print("Experiment skipped. Set RUN_HOSTED_EXPERIMENT = True only when you explicitly want a new hosted run.")
experiment_results